# Report viewer (CPU-only — no GPU, no model)

Renders results that are already computed; performs **no model computation**.
Re-derives `results/derived/summary.json` from the raw per-sample records, regenerates
`report.md` + figures, and displays them.

Run this **after** importing the Colab results (`import-results`). It is a local tool —
it does not need Colab or a GPU at all. If no real run has been imported, the same
machinery renders the PENDING state; there are no hand-typed numbers anywhere in this
project.

In [ ]:
import os
import pathlib
import subprocess
import sys

# work from the repo root whether this notebook is opened from notebooks/ or the root
# (plain os.chdir, not the %cd magic: magics inside an if-block are not portable)
if not pathlib.Path('pyproject.toml').exists():
    os.chdir('..')
assert pathlib.Path('pyproject.toml').exists(), 'run this notebook from inside the repo'
print('working dir:', os.getcwd())

# sys.executable, not bare `python`: guarantees the interpreter this kernel runs on
CLI = [sys.executable, '-m', 'rag_evidence.cli']


def run(*args: str) -> None:
    print('$', ' '.join(args))
    proc = subprocess.run([*CLI, *args], capture_output=True, text=True)
    print(proc.stdout or proc.stderr, end='')
    if proc.returncode != 0:
        raise SystemExit(f'command failed: {" ".join(args)}')


# re-derive every split that has raw records, straight from the per-sample JSONL
for cfg in ('configs/smoke.yaml', 'configs/dev.yaml', 'configs/full.yaml'):
    run('evaluate', '--config', cfg)
run('report', '--config', 'configs/full.yaml')

In [ ]:
import json
import pathlib

summary = json.loads(pathlib.Path('results/derived/summary.json').read_text(encoding='utf-8'))
print(f"generated {summary['generated_utc']} | package {summary['package_version']}")
print(f"dataset sha256:{summary['dataset_hash'][:16]}… | primary_k={summary['primary_k']}\n")

for split, payload in summary['splits'].items():
    print(f"=== {split}  (n={payload['n_questions']}) ===")
    for name, e in sorted(payload.get('generation', {}).items()):
        print(
            f"  generation {name}: EM {e['em']:.3f}  F1 {e['f1']:.3f}  "
            f"cite-F1 {e['citation']['f1']:.3f}  abstain {e['abstain_rate']:.3f}  "
            f"peak VRAM {e['peak_vram_mb']:.0f} MB  [{e['device']}]"
        )
    for mode, methods in sorted(payload.get('attribution', {}).items()):
        entries = {n: v for n, v in methods.items() if isinstance(v, dict) and 'run' in v}
        real = {n: v for n, v in entries.items() if not v['is_control']}
        ctrl = {n: v for n, v in entries.items() if v['is_control']}
        k = str(summary['primary_k'])

        def f1(e: dict) -> float:
            return e['prf_at'][k]['f1']

        if real:
            best = max(real, key=lambda n: f1(real[n]))
            worst_ctrl = max(ctrl, key=lambda n: f1(ctrl[n])) if ctrl else None
            print(
                f"  attribution [{mode}] best real: {best} F1@{k}={f1(real[best]):.3f}"
                + (
                    f"   | strongest control: {worst_ctrl} {f1(ctrl[worst_ctrl]):.3f}"
                    if worst_ctrl
                    else ''
                )
            )
        if mode == 'generated' and isinstance(methods.get('subset'), dict):
            s = methods['subset']
            print(
                f"    mode-B subset: n_correct={s['n_correct']}/{s['n_total']} "
                f"(criterion {s['criterion']}, abstained {s['n_abstained']})"
            )
    print()

In [ ]:
from IPython.display import Markdown, display
import pathlib
p = pathlib.Path('results/derived/report.md')
display(Markdown(p.read_text(encoding='utf-8'))) if p.exists() else print('no report yet')

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('assets/*.png')):
    print(f)
    display(Image(filename=f))